In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# ====================================
# LOAD DATA
# ====================================
import pandas as pd

df = pd.read_csv('pes2021-all-players.csv')

print(df.head())
print(df.columns)

In [ ]:
# ====================================
# MEMILIH FITUR
# ====================================
import numpy as np

# Kolom yang bukan atribut pemain
exclude_columns = [
    'name',
    'shirt_number',
    'team_name',
    'league',
    'nationality',
    'region',
    'age',
    'foot',
    'condition',
    'injury_resistance',
    'form',
    'com_playing_style_trickster',
    'com_playing_style_mazing_run',
    'com_playing_style_speeding_bullet',
    'com_playing_style_incisive_run',
    'com_playing_style_early_cross',
    'com_playing_style_long_ball_expert',
    'com_playing_style_long_ranger',
    'rating_as_GK',
    'rating_as_CB',
    'rating_as_LB',
    'rating_as_RB',
    'rating_as_DMF',
    'rating_as_CMF',
    'rating_as_LMF',
    'rating_as_RMF',
    'rating_as_AMF',
    'rating_as_LWF',
    'rating_as_RWF',
    'rating_as_SS',
    'rating_as_CF',
    'ball_color',
    'skill_cross_over_turn', 'skill_early_cross', 'skill_first_time_shot', 'skill_incisive_run', 'skill_one_touch_pass', 'skill_chip_shot_control', 'skill_heel_trick', 'skill_man_marking', 'skill_interception', 'skill_gk_penalty_saver', 'skill_penalty_specialist', 'skill_step_on_skill_control', 'skill_rising_shots', 'skill_marseille_turn', 'skill_gk_high_punt', 'skill_heading', 'skill_dipping_shot', 'skill_outside_curler', 'skill_captaincy', 'skill_gamesmanship', 'skill_gk_low_punt', 'skill_low_lofted_pass', 'skill_through_passing', 'skill_fighting_spirit', 'skill_flip_flap', 'skill_weighted_pass', 'skill_pinpoint_crossing', 'skill_sombrero', 'skill_acrobatic_clear', 'skill_cut_behind_turn', 'skill_long_range_drive', 'skill_scotch_move', 'skill_acrobatic_finishing', 'skill_double_touch', 'skill_gk_long_throw', 'skill_long_throw', 'skill_scissors_feint', 'skill_long_range_shooting', 'skill_track_back', 'skill_rabona', 'skill_knuckle_shot', 'skill_no_look_pass', 'skill_super_sub',
    'LWF', 'SS', 'CF', 'RWF', 'LMF', 'DMF', 'CMF', 'AMF', 'RMF', 'LB', 'CB', 'RB',
    'gk_awareness', 'gk_catching', 'gk_clearing', 'gk_reflexes', 'gk_reach'

]

feature_columns = [
    col for col in df.columns
    if col not in exclude_columns
    and np.issubdtype(df[col].dtype, np.number)
]

print(feature_columns)

In [ ]:
# ====================================
# NORMALISASI
# ====================================
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X = scaler.fit_transform(df[feature_columns])

In [ ]:
print(df.isna().sum())

# print(df[feature_columns].isna().sum())

In [ ]:
nan_cols = df[feature_columns].isna().sum()
print(nan_cols[nan_cols > 0].sort_values(ascending=False))

In [ ]:
df[feature_columns] = df[feature_columns].fillna(0)

In [ ]:
df.head(10)

In [ ]:
# ====================================
# KNN MODEL
# ====================================
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(
    n_neighbors=6,
    metric='euclidean'
)

knn.fit(X)

In [ ]:
# ====================================
# CARI PEMAIN MIRIP
# ====================================
def pemain_mirip(nama_pemain):

    pemain = df[
        df['name'].str.lower() == nama_pemain.lower()
    ]

    if pemain.empty:
        print("Pemain tidak ditemukan")
        return

    idx = pemain.index[0]

    distances, indices = knn.kneighbors(
        X[idx].reshape(1, -1)
    )

    hasil = []

    for i in range(1, len(indices[0])):

        pemain_idx = indices[0][i]

        hasil.append({
            "Nama": df.iloc[pemain_idx]["name"],
            "Kemiripan": round(
                (1/(1+distances[0][i]))*100,
                2
            )
        })

    return pd.DataFrame(hasil)

In [ ]:
pemain_mirip("C. Ronaldo")

In [ ]:
def compare_player(p1, p2):
    cols = feature_columns

    # a = df[
    #     df['name'].str.lower() == p1.lower()
    # ]
    # b = df[
    #     df['name'].str.lower() == p2.lower()
    # ]

    a = df[df['name'].str.lower() == p1.lower()][cols]
    b = df[df['name'].str.lower() == p2.lower()][cols]

    result = pd.concat([a,b])

    return result.T

In [ ]:
compare_player("C. Ronaldo", "R. LEWANDOWSKI")